# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdullah-pharmd/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Selected Lane**: **Lane 2: Refresh / Content Opportunity Scoring**

**Why this lane?**
As a Pharmacy candidate working in clinical safety and computational pharmacology, I see a direct parallel between this lane and clinical risk-stratification/decision-support systems. In clinical settings, we must prioritize which patients to screen or treat because medical resources (doctor hours, clinic beds, budgets) are strictly limited. Similarly, in SEO, content teams have a finite number of editor hours and writing budgets. They cannot review or rewrite all thousands of pages on a client's website. They need a systematic, evidence-based prioritization queue to identify which pages are declining in traffic and have the highest potential opportunity for a refresh to recover that traffic.

By predicting which high-impact pages are at risk of declining or have stale visibility, we help content editors focus their limited capacity on the most valuable opportunities, preventing wasted effort on healthy pages.

In [1]:
import pandas as pd
print("Lane selected: Lane 2 (Refresh / Content Opportunity Scoring)")


Lane selected: Lane 2 (Refresh / Content Opportunity Scoring)


## 2. The question: decision, action, cost of a wrong call

### The One-Paragraph Frame
> For **commercial client content editors and SEO strategists**, deciding **which content items (web pages) should be reviewed and refreshed first**, we will build a **ranked review queue with opportunity scores and reason codes** from **historical GSC visibility (impressions, clicks, position) and GA4 engagement metrics (sessions, scroll rate, engaged session rates)**, predicting/scoring **the likelihood of sustained traffic decline in a future 30-day window** measured by **Precision@50 and Average Precision**. A wrong call costs **wasted editor hours and content budget on healthy pages (False Positives)** or **unnoticed, compounding search traffic and revenue loss for the client (False Negatives)**. A plain rule isn't enough because **the relationships between impressions, CTR decay, average position, and freshness are non-linear, multi-dimensional, and shift across different clients and content types (e.g., comparing keyword vs. comparison articles)**. We will claim only **observed, directional, and decision-support** results.

### Key Components Breakdown
1. **What decision does this improve?**
   It improves the decision of which pages to prioritize for a content refresh or optimization from a massive inventory of thousands of pages.
2. **Who acts on the output, and what do they do?**
   Content editors, copywriters, and SEO strategists act on the output. They review the prioritized pages, update outdated information, rewrite thin content, merge competing pages, or improve search intent match.
3. **What does a wrong answer cost?**
   * **False Positives (updating a healthy page)**: Wasted editing hours and content production budget (typically $100-$300 per page). It also risks disrupting stable search rankings.
   * **False Negatives (missing a declining page)**: Sustained traffic decline, lost search visibility, reduced organic lead generation, and direct client revenue loss.
4. **Why data or ML helps at all?**
   A static heuristic (such as "refresh all pages older than 180 days") is too crude. Our starter dataset reveals that only **0.58%** of pages are older than 180 days, yet **54.21%** of pages are declining. ML allows us to combine multiple search performance and user engagement signals to capture the complex, non-linear patterns of traffic decline.

In [2]:
# Mapping the decision to a Ranking / Scoring ML task
task_type = "Ranking / Scoring"
target = "future_traffic_decline"
primary_metric = "Precision@50"
print(f"Task Type: {task_type}\nTarget: {target}\nPrimary Metric: {primary_metric}")


Task Type: Ranking / Scoring
Target: future_traffic_decline
Primary Metric: Precision@50


## 3. Quick look at the data (2-3 real numbers)

To confirm that Lane 2 is worth pursuing, we analyzed the starter dataset (`data/raw/content_refresh_anonymized.csv`) and extracted the following three real numbers:

1. **High Rate of Decline**: **16,262 pages out of 30,000 (54.21%)** in the starter dataset are classified as declining (`trend_direction == 'down'`).
2. **Significant High-Impact Opportunity**: **9,961 pages (33.20% of the entire dataset)** are both declining AND have substantial search visibility (`impressions_90d >= 500`). This is where prioritization makes a massive business difference.
3. **The Freshness Paradox (Failure of Plain Rules)**: The median time since last update is only **20 days** across the entire dataset, and only **174 pages (0.58%)** are older than 180 days. This means a simple heuristic like "refresh pages older than 180 days" would fail to detect **99.9%** of the declining pages. A multi-signal model is essential to solve this problem.

In [3]:
import pandas as pd

# Load the starter dataset
csv_path = "../../data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(csv_path)

# Number 1: Total declining pages
total_rows = len(df)
declining_rows = len(df[df['trend_direction'] == 'down'])
pct_declining = (declining_rows / total_rows) * 100
print(f"1. Declining Pages: {declining_rows} / {total_rows} ({pct_declining:.2f}%)")

# Number 2: High-impact declining pages (impressions_90d >= 500 and declining)
high_impact = len(df[(df['trend_direction'] == 'down') & (df['impressions_90d'] >= 500)])
pct_high_impact = (high_impact / total_rows) * 100
print(f"2. High-Impact Declining Pages (impressions >= 500): {high_impact} ({pct_high_impact:.2f}%)")

# Number 3: Freshness distribution vs. Decline
stale_pages = len(df[df['days_since_last_update'] >= 180])
pct_stale = (stale_pages / total_rows) * 100
print(f"3. Stale Pages (>= 180 days since update): {stale_pages} ({pct_stale:.2f}%)")
print(f"   Median days since last update: {df['days_since_last_update'].median():.1f} days")


1. Declining Pages: 16262 / 30000 (54.21%)
2. High-Impact Declining Pages (impressions >= 500): 9961 (33.20%)
3. Stale Pages (>= 180 days since update): 174 (0.58%)
   Median days since last update: 20.0 days


## 4. Careful words: what I can and can't claim

We must maintain scientific rigor and clear boundaries when analyzing search traffic data:

### What we CAN claim:
- **Observational Associations**: We can identify which observable features (e.g. CTR relative to position tier, freshness, word count) are statistically associated with a higher rate of traffic decline.
- **Directional Trend Support**: We can generate an opportunity score indicating which pages display behavioral patterns most similar to historically declining pages.
- **Decision-Support Prioritization**: We can claim that our model helps prioritize review queues, improving the efficiency of the content team by surfacing the highest-risk pages first (i.e. outperforming a random guess or simple rules on our validation sets).

### What we CANNOT claim:
- **Causal Proof**: We cannot claim that a page's traffic declined *because* of its word count or age, nor can we guarantee that updating a page will *cause* it to recover. Establishing causality would require randomized controlled trials (A/B testing) or advanced causal inference designs.
- **\"Predicting Google's Algorithm\"**: We are not reverse-engineering or predicting Google's proprietary search ranking algorithm. We are predicting our own client's traffic outcomes based on observable performance data.
- **Absolute Traffic Predictions**: We cannot predict the exact future traffic numbers or search positions of individual pages, as external factors (competitor updates, search engine algorithm updates, seasonal shifts) are unobserved and highly volatile.

In [4]:
# Defining safe observed signals and separating them from leakage risks
safe_features = ['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'word_count', 'days_since_last_update']
leaky_features = ['trend_pct', 'trend_direction', 'is_declining_label']
print("Safe Observable Signals:", safe_features)
print("Excluded Leakage Features:", leaky_features)


Safe Observable Signals: ['impressions_90d', 'clicks_90d', 'ctr', 'avg_position', 'word_count', 'days_since_last_update']
Excluded Leakage Features: ['trend_pct', 'trend_direction', 'is_declining_label']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.